In [ ]:
import json
import os

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import requests
from tqdm.notebook import tqdm

In [ ]:
with open("system-message-try3.txt") as file:
    system_message = file.read()

In [ ]:
print(system_message)

In [ ]:
testing_data = []
with open("testing-data-try3.jsonl") as file:
    for line in file:
        messages = json.loads(line)["messages"]
        assert len(messages) == 2
        assert messages[0]["role"] == "user"
        assert messages[1]["role"] == "assistant"
        user_message = messages[0]
        expected_output = json.loads(messages[1]["content"])
        testing_data.append((user_message, expected_output))

In [ ]:
len(testing_data)

In [ ]:
with open("test-results-try3.jsonl", "w") as file:
    for index, (user_message, expected_output) in tqdm(enumerate(testing_data), total=len(testing_data)):
        response = requests.post(
            "https://api.openai.com/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}",
                "Content-Type": "application/json",
            },
            json={
                "model": "ft:gpt-4.1-nano-2025-04-14:u-chicago:name-normalization-try2:Cfwhxtdm",
                "messages": [{"role": "system", "content": system_message}, user_message],
                "response_format": {
                    "type": "json_schema",
                    "json_schema": {
                        "name": "name_normalization",
                        "schema": {
                            "type": "object",
                            "properties": {
                                "Food Product Group": {"type": "string"},
                                "Food Product Category": {"type": "string"},
                                "Primary Food Product Category": {"type": "string"},
                                "Basic Type": {"type": "string"},
                                "Sub-Type": {
                                    "type": "array",
                                    "items": {"type": "string"},
                                },
                                "Flavor/Cut": {"type": "string"},
                                "Shape": {"type": "string"},
                                "Skin": {"type": "string"},
                                "Seed/Bone": {"type": "string"},
                                "Processing": {"type": "string"},
                                "Cooked/Cleaned": {"type": "string"},
                                "WG/WGR": {"type": "string"},
                                "Dietary Concern": {"type": "string"},
                                "Additives": {"type": "string"},
                                "Dietary Accommodation": {"type": "string"},
                                "Frozen": {"type": "string"},
                                "Packaging": {"type": "string"},
                                "Commodity": {"type": "string"},
                            },
                            "required": [
                                "Food Product Group",
                                "Food Product Category",
                                "Primary Food Product Category",
                            ],
                            "additionalProperties": False,
                        },
                    },
                },
            },
        )
        try:
            actual_output = json.loads(response.json().get("choices", [{}])[0].get("message", {}).get("content", "null"))
            deformed_output = None
        except:
            actual_output = None
            deformed_output = response.json()
        file.write(json.dumps({
            "index": index,
            "user_message": user_message,
            "expected_output": expected_output,
            "status_code": response.status_code,
            "error": None if response.status_code == 200 else response.json(),
            "actual_output": actual_output,
            "deformed_output": deformed_output,
        }) + "\n")

In [ ]:
columns = [
    "Food Product Group",
    "Food Product Category",
    "Primary Food Product Category",
    "Basic Type",
    "Sub-Type",
    "Flavor/Cut",
    "Shape",
    "Skin",
    "Seed/Bone",
    "Processing",
    "Cooked/Cleaned",
    "WG/WGR",
    "Dietary Concern",
    "Additives",
    "Dietary Accommodation",
    "Frozen",
    "Packaging",
    "Commodity",
]
column_data = {f"want_{c}": [] for c in columns} | {f"got_{c}": [] for c in columns}

with open("test-results-try3.jsonl") as file:
    for line in file:
        data = json.loads(line)
        if data["status_code"] != 200 or data["deformed_output"] is not None:
            continue
        expected_output = data["expected_output"]
        actual_output = data["actual_output"]
        for c in columns:
            column_data[f"want_{c}"].append(expected_output.get(c, None))
            column_data[f"got_{c}"].append(actual_output.get(c, None))

df = pd.DataFrame(column_data)

In [ ]:
df